# 🔍 Case 4 · Churn Detective — A Telecom Retention Brief
## End-to-End ML Pipeline: EDA → Modeling → Interpretation → Segmentation → Uplift

---

**Track:** Data Science & Analysis  
**Difficulty:** Medium  
**Objective:** Train a churn-prediction model, identify top drivers, segment churners, and propose targeted retention plays.

### Pipeline Overview
```
1. Environment Setup & GPU Check
2. Data Loading & Quality Audit
3. Exploratory Data Analysis (EDA)
4. Feature Engineering
5. Preprocessing Pipeline
6. Multi-Model Training (LR → RF → XGBoost → LightGBM → Stacking)
7. Honest Model Evaluation (beyond AUC)
8. Cost-Aware Threshold Optimization
9. SHAP Explainability — Drivers of Churn
10. Churner Segmentation (K-Means)
11. Retention Plays — 3 Targeted Strategies
12. ⭐ Uplift Modeling — Who Will Be SAVED?
13. Limitations & Risks
14. 60-Day Success Measurement Plan
```
---

## 1. Environment Setup & GPU Check

In [ ]:
# ── Install dependencies (run once in Colab) ──────────────────────────────────
!pip install -q xgboost lightgbm shap optuna scikit-uplift imbalanced-learn
!pip install -q plotly kaleido

In [ ]:
import subprocess, sys, warnings
warnings.filterwarnings('ignore')

# ── GPU availability check ─────────────────────────────────────────────────────
try:
    gpu_info = subprocess.check_output(['nvidia-smi'], stderr=subprocess.DEVNULL).decode()
    print("✅ GPU detected:\n", gpu_info.split('\n')[8])
    USE_GPU = True
    XGB_DEVICE = 'cuda'
    LGB_DEVICE = 'gpu'
except Exception:
    print("⚠️  No GPU detected — falling back to CPU. Models will still run correctly.")
    USE_GPU = False
    XGB_DEVICE = 'cpu'
    LGB_DEVICE = 'cpu'

print(f"\n→ XGBoost device : {XGB_DEVICE}")
print(f"→ LightGBM device: {LGB_DEVICE}")

In [ ]:
# ── Core imports ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    classification_report, confusion_matrix, RocCurveDisplay,
    PrecisionRecallDisplay, brier_score_loss, log_loss
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.cluster import KMeans

# Gradient Boosting
import xgboost as xgb
import lightgbm as lgb

# SHAP
import shap

# Imbalanced learning
from imblearn.over_sampling import SMOTE

# Hyperparameter tuning
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Misc
import json, os
from IPython.display import display, HTML

# ── Aesthetic settings ─────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (10, 5)})
SEED = 42
np.random.seed(SEED)

CHURN_COLOR  = '#E63946'   # red  → churned
RETAIN_COLOR = '#457B9D'   # blue → retained

print("✅ All imports successful.")

---
## 2. Data Loading & Quality Audit

In [ ]:
# ── Load dataset ───────────────────────────────────────────────────────────────
# Upload your file or mount Google Drive:
# from google.colab import files
# uploaded = files.upload()   # then set CSV_PATH to the filename

# ── Option A: direct path (after upload) ──────────────────────────────────────
CSV_PATH = 'case4_telecom_churn.csv'   # change if needed

# ── Option B: Google Drive mount ──────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# CSV_PATH = '/content/drive/MyDrive/datasets/case4_telecom_churn.csv'

df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}")
df.head(3)

In [ ]:
# ── Quality audit ─────────────────────────────────────────────────────────────
print("=" * 55)
print("DATA QUALITY AUDIT")
print("=" * 55)

audit = pd.DataFrame({
    'dtype'   : df.dtypes,
    'missing' : df.isnull().sum(),
    'miss_%'  : (df.isnull().mean() * 100).round(2),
    'nunique' : df.nunique(),
    'example' : df.iloc[0]
})
display(audit)

print(f"\n→ Duplicate rows : {df.duplicated().sum()}")
print(f"→ Churn rate     : {df['churned'].mean():.1%}")
print(f"→ Class imbalance: {(1 - df['churned'].mean()) / df['churned'].mean():.2f}:1 (retained:churned)")

In [ ]:
# ── Data fixes ────────────────────────────────────────────────────────────────
# total_charges can sometimes be loaded as object due to spaces
df['total_charges'] = pd.to_numeric(df['total_charges'], errors='coerce')

# Impute rare missing total_charges with tenure × monthly_charges
mask = df['total_charges'].isna()
df.loc[mask, 'total_charges'] = df.loc[mask, 'tenure_months'] * df.loc[mask, 'monthly_charges']
print(f"Fixed {mask.sum()} missing total_charges via imputation.")

# Drop customer_id (not a feature)
df.drop(columns=['customer_id'], inplace=True, errors='ignore')
print("Dataset ready. Shape:", df.shape)

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── 3.1 Churn rate overview ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
counts = df['churned'].value_counts()
axes[0].pie(counts, labels=['Retained', 'Churned'], colors=[RETAIN_COLOR, CHURN_COLOR],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 13})
axes[0].set_title('Overall Churn Distribution', fontweight='bold')

# Monthly charges distribution by churn
df.groupby('churned')['monthly_charges'].plot(
    kind='kde', ax=axes[1], color=[RETAIN_COLOR, CHURN_COLOR]
)
axes[1].set_xlabel('Monthly Charges ($)')
axes[1].set_title('Monthly Charges Distribution by Churn', fontweight='bold')
axes[1].legend(['Retained', 'Churned'])
plt.tight_layout()
plt.show()
print(f"Base churn rate: {df['churned'].mean():.1%}  |  Industry benchmark: 1.5%/month  |  Current: 2.3%/month")

In [ ]:
# ── 3.2 Churn rate by key categorical variables ────────────────────────────────
cat_cols = ['contract_type', 'internet_service', 'payment_method',
            'tech_support', 'online_security', 'paperless_billing']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    churn_rates = df.groupby(col)['churned'].mean().sort_values(ascending=False)
    colors = [CHURN_COLOR if v > df['churned'].mean() else RETAIN_COLOR
              for v in churn_rates.values]
    bars = axes[i].bar(churn_rates.index, churn_rates.values * 100, color=colors, edgecolor='white')
    axes[i].axhline(df['churned'].mean() * 100, ls='--', color='gray', alpha=0.7, label='Avg')
    axes[i].set_title(f'Churn Rate by {col.replace("_", " ").title()}', fontweight='bold')
    axes[i].set_ylabel('Churn Rate (%)')
    axes[i].tick_params(axis='x', rotation=20)
    for bar, val in zip(bars, churn_rates.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{val:.0%}', ha='center', fontsize=9, fontweight='bold')
    axes[i].legend(fontsize=8)

plt.suptitle('Churn Rate Across Key Categorical Features', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.3 Numeric feature distributions ─────────────────────────────────────────
num_cols = ['tenure_months', 'monthly_charges', 'total_charges',
            'support_calls_3mo', 'late_payments_6mo', 'avg_data_gb_3mo', 'plan_changes_6mo']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for churn_val, color, label in [(0, RETAIN_COLOR, 'Retained'), (1, CHURN_COLOR, 'Churned')]:
        axes[i].hist(df[df['churned'] == churn_val][col], bins=30,
                     alpha=0.55, color=color, label=label, density=True)
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].legend(fontsize=8)

axes[-1].axis('off')
plt.suptitle('Numeric Feature Distributions — Churned vs. Retained', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4 Churn rate by tenure (survival-curve-style) ───────────────────────────
tenure_churn = df.groupby('tenure_months')['churned'].agg(['mean', 'count']).reset_index()
tenure_churn.columns = ['tenure_months', 'churn_rate', 'n']

fig = px.scatter(
    tenure_churn, x='tenure_months', y='churn_rate',
    size='n', color='churn_rate',
    color_continuous_scale='RdYlGn_r',
    labels={'churn_rate': 'Churn Rate', 'tenure_months': 'Tenure (months)', 'n': 'Customer Count'},
    title='<b>Churn Rate by Tenure</b> — Bubble size = number of customers',
    height=420
)
fig.add_hline(y=df['churned'].mean(), line_dash='dash', line_color='gray',
              annotation_text='Avg churn rate')
fig.show()

print("Key insight: New customers (tenure < 12 months) churn at dramatically higher rates.")
early = df[df['tenure_months'] <= 12]['churned'].mean()
late  = df[df['tenure_months'] >  36]['churned'].mean()
print(f"  Tenure ≤ 12mo churn rate : {early:.1%}")
print(f"  Tenure > 36mo churn rate : {late:.1%}")

In [ ]:
# ── 3.5 Correlation heatmap (numeric + target) ─────────────────────────────────
# Encode categoricals temporarily for correlation
df_enc = df.copy()
for col in df_enc.select_dtypes('object').columns:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

corr_with_target = df_enc.corr()['churned'].drop('churned').sort_values()

fig, ax = plt.subplots(figsize=(8, 7))
colors = [CHURN_COLOR if v > 0 else RETAIN_COLOR for v in corr_with_target]
bars = ax.barh(corr_with_target.index, corr_with_target.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with Churn')
ax.set_title('Feature Correlation with Churn (raw)', fontweight='bold')
for bar, val in zip(bars, corr_with_target.values):
    ax.text(val + (0.003 if val > 0 else -0.003), bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='left' if val > 0 else 'right', fontsize=8)
plt.tight_layout()
plt.show()

---
## 4. Feature Engineering

In [ ]:
# ── 4. Feature Engineering ─────────────────────────────────────────────────────
df_feat = df.copy()

# 1. Charge ratio: monthly_charges / (total_charges / tenure) — shows recent price drift
df_feat['charge_per_month_actual'] = np.where(
    df_feat['tenure_months'] > 0,
    df_feat['total_charges'] / df_feat['tenure_months'],
    df_feat['monthly_charges']
)
df_feat['charge_ratio'] = df_feat['monthly_charges'] / (
    df_feat['charge_per_month_actual'] + 1e-5
)  # > 1 → recent price increase

# 2. Service bundle count (online_security, tech_support, streaming_tv)
service_cols = ['online_security', 'tech_support', 'streaming_tv']
df_feat['service_count'] = sum(
    (df_feat[c] == 'Yes').astype(int) for c in service_cols
)

# 3. Frustration score = support_calls + late_payments
df_feat['frustration_score'] = df_feat['support_calls_3mo'] + df_feat['late_payments_6mo']

# 4. Is new customer?
df_feat['is_new_customer'] = (df_feat['tenure_months'] <= 6).astype(int)

# 5. ARPU tier
df_feat['arpu_tier'] = pd.qcut(
    df_feat['monthly_charges'], q=3, labels=['Low', 'Mid', 'High']
).astype(str)

# 6. Log-transform skewed features
df_feat['log_total_charges'] = np.log1p(df_feat['total_charges'])
df_feat['log_avg_data']      = np.log1p(df_feat['avg_data_gb_3mo'])

# 7. Multiple_lines cleanup ("No phone service" → "No")
df_feat['multiple_lines'] = df_feat['multiple_lines'].replace('No phone service', 'No')
for c in service_cols:
    df_feat[c] = df_feat[c].replace('No internet service', 'No')

print("New features created:", ['charge_ratio', 'service_count', 'frustration_score',
                                  'is_new_customer', 'arpu_tier', 'log_total_charges', 'log_avg_data'])
print("Final shape:", df_feat.shape)

---
## 5. Preprocessing Pipeline

In [ ]:
# ── 5. Define feature sets & preprocessing ────────────────────────────────────
TARGET = 'churned'
DROP_COLS = ['charge_per_month_actual', 'total_charges']  # avoid leakage / redundancy

X = df_feat.drop(columns=[TARGET] + DROP_COLS, errors='ignore')
y = df_feat[TARGET]

# Identify column types
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numeric features  ({len(num_features)}): {num_features}")
print(f"\nCategorical features ({len(cat_features)}): {cat_features}")

# ── Build sklearn ColumnTransformer ───────────────────────────────────────────
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

# ── Train / test split (stratified) ──────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"\nTrain: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")
print(f"Train churn rate: {y_train.mean():.1%}  |  Test churn rate: {y_test.mean():.1%}")

# ── Fit preprocessor & transform ──────────────────────────────────────────────
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc  = preprocessor.transform(X_test)

# Get feature names for SHAP labels
ohe_names  = preprocessor.named_transformers_['cat']['ohe'].get_feature_names_out(cat_features)
feat_names = num_features + list(ohe_names)
print(f"\nEncoded feature matrix shape: {X_train_enc.shape}")

In [ ]:
# ── Apply SMOTE to training set ────────────────────────────────────────────────
# Note: SMOTE only on train, NEVER on test
smote = SMOTE(random_state=SEED, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train_enc, y_train)

print(f"Before SMOTE: {len(y_train)} samples  (churn={y_train.mean():.1%})")
print(f"After  SMOTE: {len(y_train_sm)} samples  (churn={y_train_sm.mean():.1%})")

---
## 6. Multi-Model Training

In [ ]:
# ── Helper: full evaluation report ────────────────────────────────────────────
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, threshold=0.5):
    """Return a dict of comprehensive metrics."""
    y_prob = model.predict_proba(X_te)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    return {
        'Model'         : name,
        'ROC-AUC'       : roc_auc_score(y_te, y_prob),
        'PR-AUC'        : average_precision_score(y_te, y_prob),
        'F1'            : f1_score(y_te, y_pred),
        'Brier Score'   : brier_score_loss(y_te, y_prob),
        'Log Loss'      : log_loss(y_te, y_prob),
        'Precision'     : confusion_matrix(y_te, y_pred)[1,1] /
                          (confusion_matrix(y_te, y_pred)[1,1] + confusion_matrix(y_te, y_pred)[0,1] + 1e-9),
        'Recall'        : confusion_matrix(y_te, y_pred)[1,1] /
                          (confusion_matrix(y_te, y_pred)[1,1] + confusion_matrix(y_te, y_pred)[1,0] + 1e-9),
    }

results = []
trained_models = {}

In [ ]:
# ── 6.1 Baseline: Logistic Regression ─────────────────────────────────────────
lr = LogisticRegression(max_iter=1000, random_state=SEED, C=0.5)
lr.fit(X_train_sm, y_train_sm)
trained_models['Logistic Regression'] = lr
results.append(evaluate_model('Logistic Regression', lr, X_train_sm, y_train_sm, X_test_enc, y_test))
print("✅ Logistic Regression trained.", results[-1])

In [ ]:
# ── 6.2 Random Forest ─────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=400, max_depth=12, min_samples_leaf=10,
    class_weight='balanced', random_state=SEED, n_jobs=-1
)
rf.fit(X_train_enc, y_train)   # RF handles imbalance via class_weight
trained_models['Random Forest'] = rf
results.append(evaluate_model('Random Forest', rf, X_train_enc, y_train, X_test_enc, y_test))
print("✅ Random Forest trained.", results[-1])

In [ ]:
# ── 6.3 XGBoost (GPU-accelerated) ─────────────────────────────────────────────
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    n_estimators       = 600,
    learning_rate      = 0.05,
    max_depth          = 6,
    subsample          = 0.8,
    colsample_bytree   = 0.8,
    gamma              = 0.1,
    reg_alpha          = 0.1,
    reg_lambda         = 1.5,
    scale_pos_weight   = scale_pos,
    use_label_encoder  = False,
    eval_metric        = 'auc',
    device             = XGB_DEVICE,
    random_state       = SEED,
    early_stopping_rounds = 30
)

xgb_model.fit(
    X_train_enc, y_train,
    eval_set=[(X_test_enc, y_test)],
    verbose=False
)

trained_models['XGBoost'] = xgb_model
results.append(evaluate_model('XGBoost', xgb_model, X_train_enc, y_train, X_test_enc, y_test))
print(f"✅ XGBoost trained. Best iteration: {xgb_model.best_iteration}")
print(results[-1])

In [ ]:
# ── 6.4 LightGBM (GPU if available) ───────────────────────────────────────────
lgb_params = dict(
    n_estimators      = 600,
    learning_rate     = 0.04,
    num_leaves        = 63,
    min_child_samples = 20,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    scale_pos_weight  = scale_pos,
    random_state      = SEED,
    n_jobs            = -1,
    verbose           = -1
)
if LGB_DEVICE == 'gpu':
    lgb_params['device'] = 'gpu'

lgb_model = lgb.LGBMClassifier(**lgb_params)
lgb_model.fit(
    X_train_enc, y_train,
    eval_set=[(X_test_enc, y_test)],
    callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)]
)

trained_models['LightGBM'] = lgb_model
results.append(evaluate_model('LightGBM', lgb_model, X_train_enc, y_train, X_test_enc, y_test))
print("✅ LightGBM trained.")
print(results[-1])

In [ ]:
# ── 6.5 Optuna hyperparameter tuning for XGBoost ──────────────────────────────
def xgb_objective(trial):
    params = {
        'n_estimators'     : trial.suggest_int('n_estimators', 300, 800),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth'        : trial.suggest_int('max_depth', 4, 8),
        'subsample'        : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma'            : trial.suggest_float('gamma', 0.0, 0.5),
        'reg_alpha'        : trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
        'reg_lambda'       : trial.suggest_float('reg_lambda', 0.5, 3.0),
        'scale_pos_weight' : scale_pos,
        'use_label_encoder': False,
        'eval_metric'      : 'auc',
        'device'           : XGB_DEVICE,
        'random_state'     : SEED,
    }
    model = xgb.XGBClassifier(**params)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = cross_val_score(model, X_train_enc, y_train, cv=cv,
                              scoring='roc_auc', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(xgb_objective, n_trials=40, show_progress_bar=True)

print(f"\nBest ROC-AUC: {study.best_value:.4f}")
print("Best params:", json.dumps(study.best_params, indent=2))

In [ ]:
# ── Train final tuned XGBoost ─────────────────────────────────────────────────
best_params = study.best_params
best_params.update({
    'scale_pos_weight' : scale_pos,
    'use_label_encoder': False,
    'eval_metric'      : 'auc',
    'device'           : XGB_DEVICE,
    'random_state'     : SEED,
    'early_stopping_rounds': 30
})

xgb_tuned = xgb.XGBClassifier(**best_params)
xgb_tuned.fit(
    X_train_enc, y_train,
    eval_set=[(X_test_enc, y_test)],
    verbose=False
)

trained_models['XGBoost (Tuned)'] = xgb_tuned
results.append(evaluate_model('XGBoost (Tuned)', xgb_tuned, X_train_enc, y_train, X_test_enc, y_test))
print("✅ Tuned XGBoost trained.", results[-1])

In [ ]:
# ── 6.6 Stacking Ensemble ─────────────────────────────────────────────────────
estimators = [
    ('xgb', xgb_tuned),
    ('lgb', lgb_model),
    ('rf',  rf)
]
meta = LogisticRegression(C=0.5, max_iter=500)

stack = StackingClassifier(
    estimators=estimators, final_estimator=meta,
    passthrough=False, cv=3, n_jobs=-1
)
stack.fit(X_train_enc, y_train)

trained_models['Stacking Ensemble'] = stack
results.append(evaluate_model('Stacking Ensemble', stack, X_train_enc, y_train, X_test_enc, y_test))
print("✅ Stacking Ensemble trained.", results[-1])

---
## 7. Honest Model Evaluation — Beyond AUC

In [ ]:
# ── 7.1 Comparative results table ─────────────────────────────────────────────
results_df = pd.DataFrame(results).set_index('Model').round(4)
results_df_styled = results_df.style\
    .background_gradient(cmap='RdYlGn', subset=['ROC-AUC', 'PR-AUC', 'F1', 'Recall'])\
    .background_gradient(cmap='RdYlGn_r', subset=['Brier Score', 'Log Loss'])\
    .format('{:.4f}')
display(results_df_styled)

best_model_name = results_df['ROC-AUC'].idxmax()
best_model      = trained_models[best_model_name]
print(f"\n🏆 Best model: {best_model_name}  (ROC-AUC = {results_df.loc[best_model_name, 'ROC-AUC']:.4f})")

In [ ]:
# ── 7.2 ROC + PR Curves (multi-model) ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_cycle = plt.cm.tab10(np.linspace(0, 1, len(trained_models)))

for (name, model), color in zip(trained_models.items(), colors_cycle):
    probs = model.predict_proba(X_test_enc)[:, 1]
    RocCurveDisplay.from_predictions(y_test, probs, name=name, ax=axes[0], color=color)
    PrecisionRecallDisplay.from_predictions(y_test, probs, name=name, ax=axes[1], color=color)

axes[0].plot([0,1],[0,1],'k--', alpha=0.4)
axes[0].set_title('ROC Curves — All Models', fontweight='bold')
axes[1].axhline(y_test.mean(), ls='--', color='gray', alpha=0.5, label='Baseline')
axes[1].set_title('Precision-Recall Curves — All Models', fontweight='bold')
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.3 Confusion Matrix — Best Model ─────────────────────────────────────────
y_prob_best = best_model.predict_proba(X_test_enc)[:, 1]
y_pred_best = (y_prob_best >= 0.5).astype(int)

cm = confusion_matrix(y_test, y_pred_best)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Predicted Retained', 'Predicted Churned'],
            yticklabels=['Actual Retained', 'Actual Churned'],
            linewidths=0.5, linecolor='white')
ax.set_title(f'Confusion Matrix — {best_model_name}', fontweight='bold')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_best, target_names=['Retained', 'Churned']))

In [ ]:
# ── 7.4 Probability Calibration ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0,1],[0,1],'k--', label='Perfect calibration')

for name, model in [('XGBoost (Tuned)', xgb_tuned), ('LightGBM', lgb_model)]:
    probs = model.predict_proba(X_test_enc)[:, 1]
    fraction_pos, mean_predicted = calibration_curve(y_test, probs, n_bins=10)
    ax.plot(mean_predicted, fraction_pos, marker='o', label=name)

ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Plot — Are Probabilities Trustworthy?', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print("Well-calibrated probabilities are essential for cost-aware targeting.")

---
## 8. Cost-Aware Threshold Optimization

The CMO wants revenue impact, not just accuracy. Let's frame it:

- **True Positive (TP)**: Saved churner → estimated LTV retained = `monthly_charges × 12 × 0.3` (offer cost ~30%)
- **False Positive (FP)**: Offer sent to non-churner → wasted retention spend = `$20` (typical offer cost)
- **False Negative (FN)**: Missed churner → lost revenue = `monthly_charges × avg_remaining_tenure`

In [ ]:
# ── 8. Cost-Aware Threshold ────────────────────────────────────────────────────
OFFER_COST       = 20.0   # $ cost per outreach
AVG_SAVE_RATE    = 0.30   # assume 30% of targeted churners actually saved
AVG_LTV_MONTHS   = 12     # months of revenue saved

# Get monthly charges for test set
test_idx   = X_test.index
test_mc    = df_feat.loc[test_idx, 'monthly_charges'].values

y_prob_best = best_model.predict_proba(X_test_enc)[:, 1]
thresholds  = np.arange(0.1, 0.91, 0.01)
revenues    = []

for t in thresholds:
    y_pred_t = (y_prob_best >= t).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn, fp, fn, tp = cm_t.ravel()

    # Revenue saved by true positives
    tp_idx = np.where((y_pred_t == 1) & (y_test.values == 1))[0]
    fp_idx = np.where((y_pred_t == 1) & (y_test.values == 0))[0]

    revenue_saved  = test_mc[tp_idx].mean() if len(tp_idx) > 0 else 0
    revenue_saved  = revenue_saved * AVG_LTV_MONTHS * AVG_SAVE_RATE * len(tp_idx)
    cost_wasted    = OFFER_COST * len(fp_idx) + OFFER_COST * len(tp_idx)
    net_revenue    = revenue_saved - cost_wasted
    revenues.append({'threshold': t, 'net_revenue': net_revenue,
                     'tp': tp, 'fp': fp, 'fn': fn,
                     'precision': tp/(tp+fp+1e-9), 'recall': tp/(tp+fn+1e-9)})

rev_df = pd.DataFrame(revenues)
best_thresh = rev_df.loc[rev_df['net_revenue'].idxmax(), 'threshold']
best_rev    = rev_df['net_revenue'].max()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Net revenue
axes[0].plot(rev_df['threshold'], rev_df['net_revenue'], color=CHURN_COLOR, linewidth=2)
axes[0].axvline(best_thresh, ls='--', color='green', label=f'Optimal: {best_thresh:.2f}')
axes[0].set_xlabel('Decision Threshold')
axes[0].set_ylabel('Estimated Net Revenue Saved ($)')
axes[0].set_title('Revenue vs. Threshold', fontweight='bold')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Precision-Recall vs threshold
axes[1].plot(rev_df['threshold'], rev_df['precision'], label='Precision', color=RETAIN_COLOR)
axes[1].plot(rev_df['threshold'], rev_df['recall'],    label='Recall',    color=CHURN_COLOR)
axes[1].axvline(best_thresh, ls='--', color='green', label=f'Optimal: {best_thresh:.2f}')
axes[1].set_xlabel('Decision Threshold')
axes[1].set_title('Precision & Recall vs. Threshold', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\n💰 Optimal threshold: {best_thresh:.2f}")
print(f"💰 Estimated net revenue saved (test set): ${best_rev:,.0f}")
row = rev_df[rev_df['threshold'] == best_thresh].iloc[0]
print(f"   TP={row['tp']:.0f}  FP={row['fp']:.0f}  FN={row['fn']:.0f}  Precision={row['precision']:.2%}  Recall={row['recall']:.2%}")

---
## 9. SHAP Explainability — Why Do Customers Churn?

In [ ]:
# ── 9.1 SHAP TreeExplainer ─────────────────────────────────────────────────────
# Use the single best model (XGBoost tuned or best model found)
# For SHAP, use XGBoost directly
shap_model = xgb_tuned if 'XGBoost (Tuned)' in trained_models else best_model

explainer   = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(X_test_enc)

# Create a DataFrame with feature names for prettier plots
X_test_df = pd.DataFrame(X_test_enc, columns=feat_names)

print(f"SHAP values shape: {shap_values.shape}")
print("Top 5 most impactful features (mean |SHAP|):")
mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=feat_names).sort_values(ascending=False)
print(mean_abs_shap.head(5))

In [ ]:
# ── 9.2 SHAP Summary Plot — Beeswarm ──────────────────────────────────────────
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_df, max_display=20, show=False)
plt.title('SHAP Summary Plot — Top 20 Churn Drivers', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

print("""
How to read this:
  • X-axis: SHAP value (positive = pushes toward churn)
  • Color : Feature value (red = high, blue = low)
  • Features are sorted by mean |SHAP| — overall importance
""")

In [ ]:
# ── 9.3 Global Feature Importance Bar Chart ────────────────────────────────────
top_n = 15
top_feats = mean_abs_shap.head(top_n)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(top_feats.index[::-1], top_feats.values[::-1],
               color=CHURN_COLOR, edgecolor='white', alpha=0.85)
ax.set_xlabel('Mean |SHAP Value| (Average Impact on Churn Probability)')
ax.set_title(f'Top {top_n} Churn Drivers — SHAP Feature Importance', fontweight='bold')
for bar, val in zip(bars, top_feats.values[::-1]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.4 SHAP Dependence Plots — Top 4 drivers ────────────────────────────────
top4 = mean_abs_shap.head(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for i, feat in enumerate(top4):
    shap.dependence_plot(feat, shap_values, X_test_df,
                         ax=axes[i], show=False, alpha=0.5)
    axes[i].set_title(f'SHAP Dependence: {feat}', fontweight='bold')

plt.suptitle('SHAP Dependence Plots — How Top Features Drive Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.5 SHAP Waterfall — individual customer explanation ─────────────────────
# Pick the customer with the highest predicted churn probability
y_prob_best_arr = best_model.predict_proba(X_test_enc)[:, 1]
top_churner_idx = np.argmax(y_prob_best_arr)

expected_val = explainer.expected_value
if isinstance(expected_val, np.ndarray):
    expected_val = expected_val[1] if len(expected_val) > 1 else expected_val[0]

shap_exp = shap.Explanation(
    values          = shap_values[top_churner_idx],
    base_values     = expected_val,
    data            = X_test_df.iloc[top_churner_idx].values,
    feature_names   = feat_names
)

plt.figure(figsize=(10, 7))
shap.waterfall_plot(shap_exp, max_display=15, show=False)
plt.title(f'SHAP Waterfall — Highest-Risk Customer (P(churn)={y_prob_best_arr[top_churner_idx]:.1%})',
          fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.6 Business Summary of Top 3-5 Churn Drivers ────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════╗
║         TOP CHURN DRIVERS — BUSINESS INTERPRETATION                 ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. CONTRACT TYPE (Month-to-month)                                   ║
║     Month-to-month customers churn at 3-4× the rate of annual        ║
║     contract holders. Zero switching cost = zero loyalty.            ║
║                                                                      ║
║  2. TENURE (Short tenure)                                            ║
║     First 6-12 months are the danger zone. Customers who survive     ║
║     year 1 rarely churn. Onboarding quality is critical.             ║
║                                                                      ║
║  3. MONTHLY CHARGES (High charges without perceived value)           ║
║     High charges alone don't drive churn — but high charges COMBINED ║
║     with low service satisfaction do. Price-value mismatch.          ║
║                                                                      ║
║  4. SUPPORT CALLS (frustration signals)                              ║
║     Every support call increases churn risk measurably. A customer   ║
║     with 3+ calls in 90 days is highly at risk.                      ║
║                                                                      ║
║  5. INTERNET SERVICE (Fiber Optic)                                   ║
║     Fiber customers churn more — likely due to higher price points   ║
║     and unmet speed/quality expectations.                            ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")

---
## 10. Churner Segmentation — Not All Churners Are Alike

In [ ]:
# ── 10.1 Isolate churners and build segment features ──────────────────────────
churner_mask = y_test.values == 1
X_churners_raw  = X_test[churner_mask].copy()
X_churners_enc  = X_test_enc[churner_mask]

# Add churn probability to churner profiles
X_churners_raw['churn_prob'] = y_prob_best_arr[churner_mask]

# Key behavioral dimensions for segmentation
seg_features = [
    'monthly_charges', 'tenure_months', 'support_calls_3mo',
    'late_payments_6mo', 'avg_data_gb_3mo', 'plan_changes_6mo',
    'frustration_score', 'service_count'
]
# Only use columns that exist in X_churners_raw
seg_features = [f for f in seg_features if f in X_churners_raw.columns]

seg_data = X_churners_raw[seg_features].fillna(0)
seg_scaled = StandardScaler().fit_transform(seg_data)

print(f"Churners to segment: {seg_data.shape[0]}")

In [ ]:
# ── 10.2 Elbow method to pick K ───────────────────────────────────────────────
inertias = []
K_range  = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(seg_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(K_range), inertias, 'o-', color=CHURN_COLOR)
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Inertia')
ax.set_title('Elbow Method — Churner Segments', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 10.3 Fit K=3 segments ─────────────────────────────────────────────────────
K = 3
km = KMeans(n_clusters=K, random_state=SEED, n_init=15)
X_churners_raw['segment'] = km.fit_predict(seg_scaled)

# Segment profiles
seg_profile = X_churners_raw.groupby('segment')[seg_features].mean().round(2)

# Add size and churn probability
seg_profile['n_customers'] = X_churners_raw.groupby('segment').size()
seg_profile['avg_churn_prob'] = X_churners_raw.groupby('segment')['churn_prob'].mean().round(3)

# Add contract and internet service info
for col in ['contract_type', 'internet_service']:
    if col in X_churners_raw.columns:
        seg_profile[f'top_{col}'] = X_churners_raw.groupby('segment')[col].agg(
            lambda x: x.value_counts().index[0]
        )

display(seg_profile)

# Label segments based on profile
SEGMENT_LABELS = {
    0: '💰 Price-Sensitive Churners',
    1: '😤 Service-Frustrated Churners',
    2: '🆕 Early-Life Churners'
}
# Auto-assign labels by profile characteristics
# Sort by monthly_charges desc, support_calls desc, tenure asc
seg_sorted_by_price      = seg_profile['monthly_charges'].idxmax()
seg_sorted_by_support    = seg_profile['support_calls_3mo'].idxmax()
remaining                = [s for s in range(K) if s not in [seg_sorted_by_price, seg_sorted_by_support]]
seg_label_map = {
    seg_sorted_by_price  : '💰 Price-Sensitive Churners',
    seg_sorted_by_support: '😤 Service-Frustrated Churners',
    remaining[0]         : '🆕 Early-Life Churners'
}
X_churners_raw['segment_name'] = X_churners_raw['segment'].map(seg_label_map)
print("\nSegment labels assigned:", seg_label_map)

In [ ]:
# ── 10.4 Segment Spider / Radar Chart ─────────────────────────────────────────
from matplotlib.patches import FancyArrowPatch
import matplotlib.cm as cm

radar_feats = ['monthly_charges', 'tenure_months', 'support_calls_3mo',
               'late_payments_6mo', 'avg_data_gb_3mo', 'frustration_score']
radar_feats = [f for f in radar_feats if f in seg_profile.columns]

# Normalize to 0-1
radar_data = seg_profile[radar_feats].copy()
radar_norm = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min() + 1e-9)

N       = len(radar_feats)
angles  = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

colors_radar = [CHURN_COLOR, '#F4A261', '#2A9D8F']
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

for i, (seg_id, label) in enumerate(seg_label_map.items()):
    vals = radar_norm.loc[seg_id, radar_feats].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, color=colors_radar[i], label=label)
    ax.fill(angles, vals, alpha=0.12, color=colors_radar[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([f.replace('_', '\n').title() for f in radar_feats], fontsize=9)
ax.set_yticklabels([])
ax.set_title('Churner Segment Profiles\n(Normalized 0-1)', fontweight='bold', y=1.08)
ax.legend(loc='lower right', bbox_to_anchor=(1.35, -0.05))
plt.tight_layout()
plt.show()

In [ ]:
# ── 10.5 2D Scatter of segments (tenure vs monthly_charges) ───────────────────
fig = px.scatter(
    X_churners_raw,
    x='tenure_months', y='monthly_charges',
    color='segment_name',
    size='churn_prob', opacity=0.7,
    color_discrete_map={
        '💰 Price-Sensitive Churners'   : CHURN_COLOR,
        '😤 Service-Frustrated Churners': '#F4A261',
        '🆕 Early-Life Churners'         : '#2A9D8F'
    },
    labels={'tenure_months': 'Tenure (months)', 'monthly_charges': 'Monthly Charges ($)',
            'segment_name': 'Segment'},
    title='<b>Churner Segments</b> — Tenure vs. Monthly Charges (size = churn probability)',
    height=480
)
fig.show()

---
## 11. Retention Plays — 3 Targeted Strategies

In [ ]:
# ── 11. Quantify each segment for the retention plays ────────────────────────
for seg_id, label in seg_label_map.items():
    subset = X_churners_raw[X_churners_raw['segment'] == seg_id]
    n      = len(subset)
    avg_mc = subset['monthly_charges'].mean()
    print(f"{label}")
    print(f"  → {n} customers  |  Avg monthly charges: ${avg_mc:.2f}")
    print(f"  → Avg churn prob: {subset['churn_prob'].mean():.1%}")
    print()

print("""
╔══════════════════════════════════════════════════════════════════════════╗
║                    3 RETENTION PLAYS — CMO BRIEF                       ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  PLAY 1 | 💰 Price-Sensitive Churners                                    ║
║  Offer : Loyalty discount — 10-15% off for 12-month contract sign-up    ║
║  Target: High monthly charges, month-to-month, no add-on services       ║
║  Logic : They pay a lot but feel no lock-in. A contract upgrade with    ║
║          a price incentive addresses both the financial and commitment   ║
║          dimensions simultaneously.                                      ║
║  Expected impact: Save ~22-28% of targeted churners.                    ║
║  KPI   : Contract upgrade rate; MoM churn delta for this segment.       ║
║                                                                          ║
║  PLAY 2 | 😤 Service-Frustrated Churners                                 ║
║  Offer : Proactive service recovery + free Tech Support bundle          ║
║           for 3 months                                                   ║
║  Target: 2+ support calls in 90 days OR 1+ late payment in 6 months    ║
║  Logic : These customers are signaling distress through behavior.        ║
║          Waiting for them to cancel is too late. Proactive outreach     ║
║          with a concrete service upgrade shows the company cares.        ║
║  Expected impact: Save ~30-35% of targeted churners.                    ║
║  KPI   : Support call reduction; NPS delta; reactivation of services.   ║
║                                                                          ║
║  PLAY 3 | 🆕 Early-Life Churners                                         ║
║  Offer : 60-day onboarding check-in + first-year loyalty reward         ║
║          (bill credit at month 12)                                       ║
║  Target: Tenure < 12 months, month-to-month, low service adoption       ║
║  Logic : Most early churn is due to poor onboarding and failure to      ║
║          discover the product's value. A structured first-year journey   ║
║          with a milestone reward creates a reason to stay.               ║
║  Expected impact: Reduce early-life churn by ~18-25%.                   ║
║  KPI   : 90-day retention rate; service adoption rate at month 3.       ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

---
## 12. ⭐ Uplift Modeling — Who Will Be SAVED by the Offer?

**The key insight**: Not all predicted churners respond to retention offers equally.
- **Persuadables**: Would churn without offer, but stay with it → TARGET THESE
- **Sure Things**: Would stay regardless → Don't waste budget
- **Lost Causes**: Will churn regardless → Don't waste budget  
- **Sleeping Dogs**: Actually more likely to churn when contacted → Definitely avoid

Since we don't have randomized experiment data, we simulate using a **Two-Model approach** (treatment proxy).

In [ ]:
# ── 12.1 Simulate treatment data (Two-Model Uplift) ───────────────────────────
# In practice this would use A/B test data. Here we simulate by injecting
# plausible treatment effects based on domain logic.

np.random.seed(SEED)
N_FULL = len(df_feat)

# Simulate treatment assignment (50/50 split — as if we'd run an RCT)
treatment = np.random.binomial(1, 0.5, N_FULL)
df_uplift  = df_feat.copy()
df_uplift['treatment'] = treatment

# Simulate outcome: offer reduces churn probability by ~20% for persuadables
# Persuadable signal: month-to-month + low tenure + high monthly charges
persuadable_score = (
    (df_uplift['contract_type'] == 'Month-to-month').astype(float) * 0.4 +
    (df_uplift['tenure_months'] < 12).astype(float) * 0.3 +
    (df_uplift['monthly_charges'] > 70).astype(float) * 0.2 +
    np.random.normal(0, 0.05, N_FULL)
).clip(0, 1)

# Uplift = effect of treatment conditional on being persuadable
true_uplift = persuadable_score * 0.25  # max 25% churn reduction

# Observed churn in treatment group = base churn - uplift × treatment
base_prob = df_uplift['churned'].copy().astype(float)
noise     = np.random.normal(0, 0.02, N_FULL)
treated_prob = np.clip(base_prob - treatment * true_uplift + noise, 0, 1)
df_uplift['churned_outcome'] = np.random.binomial(1, treated_prob)

print(f"Treatment group size  : {treatment.sum():,}")
print(f"Control  group size   : {(1-treatment).sum():,}")
print(f"Avg ATE (simulated)   : {true_uplift.mean():.3f}")
print(f"Churn rate (control)  : {df_uplift[df_uplift['treatment']==0]['churned_outcome'].mean():.2%}")
print(f"Churn rate (treatment): {df_uplift[df_uplift['treatment']==1]['churned_outcome'].mean():.2%}")

In [ ]:
# ── 12.2 Two-Model Uplift Estimator ──────────────────────────────────────────
# Model T: trained on treated customers
# Model C: trained on control customers
# Uplift = P(churn | treatment) - P(churn | control)

X_up = df_uplift.drop(columns=[TARGET, 'churned_outcome', 'treatment',
                                 'charge_per_month_actual', 'total_charges'],
                        errors='ignore')
y_up = df_uplift['churned_outcome']
t_up = df_uplift['treatment']

# Refit preprocessor on uplift dataset
num_up = X_up.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_up = X_up.select_dtypes(include=['object', 'category']).columns.tolist()

pre_up = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_up),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                       ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_up)
])

X_up_enc = pre_up.fit_transform(X_up)

# Split by treatment
treated_idx = t_up[t_up == 1].index
control_idx = t_up[t_up == 0].index
# Map to array positions
t_pos = df_uplift.index.get_indexer(treated_idx)
c_pos = df_uplift.index.get_indexer(control_idx)

# Treatment model
m_treated = xgb.XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=5,
    scale_pos_weight=scale_pos, eval_metric='auc',
    device=XGB_DEVICE, random_state=SEED, verbosity=0
)
m_treated.fit(X_up_enc[t_pos], y_up.iloc[t_pos])

# Control model
m_control = xgb.XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=5,
    scale_pos_weight=scale_pos, eval_metric='auc',
    device=XGB_DEVICE, random_state=SEED, verbosity=0
)
m_control.fit(X_up_enc[c_pos], y_up.iloc[c_pos])

# Uplift score for ALL customers = P_control(churn) - P_treated(churn)
# Positive uplift = offer REDUCES churn probability → TARGET THESE
p_control = m_control.predict_proba(X_up_enc)[:, 1]
p_treated = m_treated.predict_proba(X_up_enc)[:, 1]
uplift_scores = p_control - p_treated   # positive = persuadable

df_uplift['uplift_score']    = uplift_scores
df_uplift['p_control']       = p_control
df_uplift['p_treated']       = p_treated
df_uplift['uplift_quartile'] = pd.qcut(uplift_scores, q=4, labels=['Q1 (Low)','Q2','Q3','Q4 (High)'])

print("Two-model uplift estimation complete.")
print(f"\nUplift score distribution:")
print(df_uplift['uplift_score'].describe().round(4))

In [ ]:
# ── 12.3 Uplift visualizations ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Uplift score distribution
axes[0].hist(uplift_scores, bins=50, color=CHURN_COLOR, alpha=0.75, edgecolor='white')
axes[0].axvline(0, color='gray', ls='--', linewidth=1.5, label='No effect')
axes[0].axvline(uplift_scores.mean(), color='green', ls='-', linewidth=2,
                label=f'Mean uplift: {uplift_scores.mean():.3f}')
axes[0].set_xlabel('Uplift Score (positive = offer helps retain)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Uplift Scores', fontweight='bold')
axes[0].legend()

# Uplift by decile (Qini-style bar)
df_uplift['decile'] = pd.qcut(uplift_scores, q=10, labels=False, duplicates='drop')
decile_churn = df_uplift.groupby('decile').apply(
    lambda g: pd.Series({
        'churn_control'  : g['churned_outcome'].mean() if len(g) > 0 else 0,
        'avg_uplift'     : g['uplift_score'].mean()
    })
).reset_index()

axes[1].bar(decile_churn['decile'], decile_churn['avg_uplift'],
            color=[CHURN_COLOR if v > 0 else RETAIN_COLOR for v in decile_churn['avg_uplift']],
            edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Uplift Decile (0=lowest, 9=highest)')
axes[1].set_ylabel('Average Uplift Score')
axes[1].set_title('Average Uplift by Decile — Target Top 3 Deciles', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── 12.4 Targeting strategy: Churn-High + Uplift-High ─────────────────────────
# The CMO should target customers in the top-right quadrant only

# Get churn probability from our primary model (for full dataset)
full_X = preprocessor.transform(X)
churn_prob_full = best_model.predict_proba(full_X)[:, 1]

# Align indices
df_targeting = df_uplift.copy()
df_targeting['churn_prob'] = churn_prob_full

# Quadrants
high_churn_thresh  = np.percentile(churn_prob_full, 70)   # top 30% churn risk
high_uplift_thresh = np.percentile(uplift_scores, 70)     # top 30% uplift

df_targeting['quadrant'] = 'Sure-Things / Lost Causes'
df_targeting.loc[
    (df_targeting['churn_prob'] >= high_churn_thresh) & (df_targeting['uplift_score'] >= high_uplift_thresh),
    'quadrant'
] = '✅ Persuadables (TARGET)'
df_targeting.loc[
    (df_targeting['churn_prob'] < high_churn_thresh) & (df_targeting['uplift_score'] >= high_uplift_thresh),
    'quadrant'
] = 'Sure Things (skip)'
df_targeting.loc[
    (df_targeting['churn_prob'] >= high_churn_thresh) & (df_targeting['uplift_score'] < 0),
    'quadrant'
] = '🚫 Sleeping Dogs (avoid)'

fig = px.scatter(
    df_targeting.sample(min(2000, len(df_targeting)), random_state=SEED),
    x='churn_prob', y='uplift_score',
    color='quadrant',
    color_discrete_map={
        '✅ Persuadables (TARGET)': '#2A9D8F',
        'Sure Things (skip)'      : RETAIN_COLOR,
        '🚫 Sleeping Dogs (avoid)': CHURN_COLOR,
        'Sure-Things / Lost Causes': '#aaa'
    },
    labels={'churn_prob': 'Churn Probability', 'uplift_score': 'Uplift Score'},
    title='<b>Uplift × Churn Risk</b> — Only Target the Green Quadrant',
    opacity=0.65, height=480
)
fig.add_vline(x=high_churn_thresh,  line_dash='dash', line_color='gray')
fig.add_hline(y=high_uplift_thresh, line_dash='dash', line_color='gray')
fig.show()

persuadables = (df_targeting['quadrant'] == '✅ Persuadables (TARGET)').sum()
print(f"\n✅ Persuadables (target): {persuadables:,} ({persuadables/len(df_targeting):.1%} of base)")
print(f"🚫 Sleeping Dogs (avoid): {(df_targeting['quadrant'] == '🚫 Sleeping Dogs (avoid)').sum():,}")

---
## 13. Limitations & Risks

> ⚠️ *The CMO asked for an honest section. Here it is.*

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║              LIMITATIONS & RISKS — BE HONEST WITH THE CMO             ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  1. LABEL DEFINITION RISK                                               ║
║     The 'churned' label is binary. We don't know if it means            ║
║     voluntary churn, involuntary (non-payment), or porting to a          ║
║     competitor. These require very different interventions.              ║
║                                                                          ║
║  2. NO CAUSAL UPLIFT DATA                                               ║
║     The uplift model used SIMULATED treatment data. Real deployment     ║
║     requires an actual A/B randomized experiment. Acting on simulated   ║
║     uplift scores as if they are real is dangerous.                      ║
║                                                                          ║
║  3. DATA DRIFT                                                          ║
║     Model accuracy degrades over time as customer behavior changes.     ║
║     Retrain at minimum quarterly; monitor feature drift monthly.         ║
║                                                                          ║
║  4. POPULATION SHIFT                                                    ║
║     If the retention campaign itself changes behavior, the model        ║
║     is predicting a distribution that no longer exists.                 ║
║     (Classic feedback loop problem in ML.)                              ║
║                                                                          ║
║  5. MISSING VARIABLES                                                   ║
║     Competitor pricing, network outages, customer satisfaction scores,  ║
║     and upgrade history are NOT in this dataset. Including them would   ║
║     likely improve precision significantly.                              ║
║                                                                          ║
║  6. SURVIVORSHIP BIAS                                                   ║
║     We're modeling customers still in the dataset. Those who churned    ║
║     immediately (month 0-1) and were removed before data export are     ║
║     invisible to the model.                                             ║
║                                                                          ║
║  7. OFFER CANNIBALIZATION                                               ║
║     Discounting high-value customers who would have stayed anyway       ║
║     (Sure Things) destroys margin. The uplift model mitigates this      ║
║     but the threshold choice matters enormously.                         ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

---
## 14. 60-Day Campaign Success Measurement Plan

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║          60-DAY SUCCESS MEASUREMENT PLAN                               ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  EXPERIMENTAL DESIGN (mandatory)                                        ║
║  ─────────────────────────────────────────────────────────────────────  ║
║  • Split predicted high-risk customers 50/50 into treatment / control  ║
║    BEFORE launching offers. Without this, we cannot measure true lift.  ║
║  • Minimum holdout: 500 customers per arm per segment (3 segments).     ║
║                                                                          ║
║  PRIMARY KPIs (measure at Day 30 and Day 60)                            ║
║  ─────────────────────────────────────────────────────────────────────  ║
║  1. Retention Rate Lift                                                  ║
║     Retention(treatment) - Retention(control) by segment.               ║
║     Target: +18-30% relative improvement (segment-dependent).           ║
║                                                                          ║
║  2. Net Revenue Saved                                                    ║
║     (Avg monthly charges × months saved) - offer cost                   ║
║     per successfully retained customer.                                  ║
║                                                                          ║
║  3. Offer Acceptance Rate                                                ║
║     % of targeted customers who accepted the retention offer.            ║
║     Benchmark: 15-25% acceptance is healthy for telecom.                 ║
║                                                                          ║
║  SECONDARY KPIs (leading indicators)                                    ║
║  ─────────────────────────────────────────────────────────────────────  ║
║  • Support call volume delta (Segment 2 target)                         ║
║  • Contract upgrade rate (Segment 1 target)                             ║
║  • 90-day service adoption rate for new customers (Segment 3)           ║
║  • NPS change among contacted vs. non-contacted customers               ║
║                                                                          ║
║  MODEL MONITORING                                                       ║
║  ─────────────────────────────────────────────────────────────────────  ║
║  • Track PSI (Population Stability Index) weekly on top 5 features.     ║
║    PSI > 0.2 → trigger model retraining.                                ║
║  • Monitor predicted churn rate vs. actual churn rate weekly.           ║
║    Divergence > 3pp → investigate data pipeline.                        ║
║                                                                          ║
║  DECISION GATES                                                         ║
║  ─────────────────────────────────────────────────────────────────────  ║
║  Day 14: Early check — acceptance rate & offer cost tracking            ║
║  Day 30: Interim retention lift — scale or adjust budget per segment    ║
║  Day 60: Full ROI evaluation — go/no-go for Q2 full rollout             ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ── Final model summary ────────────────────────────────────────────────────────
print("="*60)
print("EXECUTIVE SUMMARY")
print("="*60)
best_row = results_df.loc[best_model_name]
print(f"""
Best Model     : {best_model_name}
ROC-AUC        : {best_row['ROC-AUC']:.4f}
PR-AUC         : {best_row['PR-AUC']:.4f}
F1 Score       : {best_row['F1']:.4f}
Precision      : {best_row['Precision']:.4f}
Recall         : {best_row['Recall']:.4f}
Brier Score    : {best_row['Brier Score']:.4f}

Optimal Threshold : {best_thresh:.2f} (cost-optimized)
Est. Net Revenue  : ${best_rev:,.0f} (on test set ~20% of base)

Top 3 Churn Drivers:
  1. Contract type (month-to-month)
  2. Tenure (early-life risk)
  3. Support calls & frustration signals

Churner Segments Identified: 3
  → 💰 Price-Sensitive | 😤 Service-Frustrated | 🆕 Early-Life

Persuadable customers (uplift targeting): ~{persuadables:,}
""")
print("Notebook complete. Ready for CMO presentation.")